# PowerNet: A Learning power allocation in wireless networks

> PowerNet is an MLP-based architecture for learning power allocation in wireless networks. It is designed to be simple and efficient, while still achieving state-of-the-art performance on the problem of power allocation in wireless networks.

In [ ]:
#| default_exp models.power_net

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import torch
from torch import nn
from torch.nn import functional as F


In [1]:
#| export
import torch
import torch.nn as nn


In [2]:
#| export
lin_layer = nn.Linear(2, 20)
inp = torch.randn(5, 2)
out = lin_layer(inp)
out.shape

torch.Size([5, 20])

In [ ]:
#| export 

class BatchNorm1d(nn.Module):
    def __init__(self, num_features: int):
        super(BatchNorm1d, self).__init__()
        self.num_features = num_features
        self.gamma = nn.Parameter(torch.ones(num_features))
        self.beta = nn.Parameter(torch.zeros(num_features))
        self.eps = 1e-5
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass of the BatchNorm1d layer.
        
        Args:
            x (torch.Tensor): A tensor of shape (batch_size, num_features) representing the input features.
        
        Returns:
            torch.Tensor: A tensor of shape (batch_size, num_features) representing the normalized features.
        """
        mean = x.mean(dim=0)
        var = x.var(dim=0, unbiased=False)
        
        x_normalized = (x - mean) / torch.sqrt(var + self.eps)
        out = self.gamma * x_normalized + self.beta
        
        return out



class CSIEncoder(nn.Module):
    def __init__(self, csi_dim: int, hidden_size: int = 128):
        super(CSIEncoder, self).__init__()
        self.csi_dim = csi_dim
        self.hidden_size = hidden_size
        
        # Define the MLP layers for CSI encoding
        self.fc1 = nn.Linear(csi_dim, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        
    def forward(self, csi: torch.Tensor) -> torch.Tensor:
        """
        Forward pass of the CSIEncoder model.
        
        Args:
            csi (torch.Tensor): A tensor of shape (batch_size, num_users, num_users) representing the channel state information between users.
        
        Returns:
            torch.Tensor: A tensor of shape (batch_size, hidden_size) representing the encoded CSI features.
        """
        batch_size = csi.size(0)
        
        # Flatten the CSI to feed into the MLP
        x = csi.view(batch_size, -1)
        
        # Pass through the MLP layers with ReLU activations
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        
        return x
    


class PowerNet(nn.Module):
    def __init__(self, csi_dim: int, hidden_size: int = 128):
        super(PowerNet, self).__init__()
        self.csi_dim = csi_dim
        self.hidden_size = hidden_size
        
        # Define the MLP layers
        self.csi_encoder = nn.Linear(csi_dim, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, num_users)
        
    def forward(self, channel_gains: torch.Tensor) -> torch.Tensor:
        """
        Forward pass of the PowerNet model.
        
        Args:
            channel_gains (torch.Tensor): A tensor of shape (batch_size, num_users, num_users) representing the channel gains between users.
        
        Returns:
            torch.Tensor: A tensor of shape (batch_size, num_users) representing the power allocation for each user.
        """
        batch_size = channel_gains.size(0)
        
        # Flatten the channel gains to feed into the MLP
        x = channel_gains.view(batch_size, -1)
        
        # Pass through the MLP layers with ReLU activations
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        
        # Output layer with sigmoid activation to ensure power allocation is between 0 and 1
        power_allocation = torch.sigmoid(self.fc3(x))
        
        return power_allocation

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()